In [ ]:
import pandas as pd
df = pd.read_json("test.jsonl", lines=True)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
from src.codec.translate import labels_to_bracketed

ex = df.iloc[0]
bracketed, labs = labels_to_bracketed(ex.text, ex.label)
print(ex.source_text)
print(bracketed)

Bedingt durch den demografischen Wandel sieht sich die junge Generation verstärkt finanziellen Belastungen ausgesetzt.
Due to demographic changes, [the younger generation] is increasingly exposed to financial burdens.


In [3]:
from src.codec.translate import bracketed_to_label_offsets

print(labs)
bracketed_to_label_offsets(bracketed, entity_type="TEST")

[[28, 50, 'SG']]


[28, 50, 'TEST']

In [4]:
from src.codec.codec import Codec

codec = Codec(
    model_name_or_path="ychenNLP/nllb-200-distilled-1.3B-easyproject",
    tokenizer_path="facebook/nllb-200-distilled-1.3B",
    src_lang="eng_Latn",
    # tgt_lang="deu_Latn",
    mt_name='nllb',
    max_length=256,
)

/home/hauke-licht/miniforge/envs/codec/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/hauke-licht/miniforge/envs/codec/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [56]:
translate_kwargs = dict(
    search_mode=0,
    batch_size=8,
    future_steps=-1,
    n_best=1,  # we only need the top-1 result
)

In [ ]:
text = "[Older people], [women], [people with a migration background] and [low-skilled people]."
target = "Ältere, Frauen, Menschen mit Migrationshintergrund und Geringqualifizierte."

output = codec.translate(
    src_text=text,
    template=target,
    n_spans=4,
    tgt_lang="deu_Latn",
    **translate_kwargs
)

In [68]:
df['n_labels_'] = df.label.apply(len)
grps = iter(df.groupby('n_labels_'))
n_, subdf = next(grps)
n_, subdf = next(grps)
n_, subdf = next(grps)
n_ 

2

In [ ]:
bracketed, labs = labels_to_bracketed(ex.text, ex.label)

In [62]:
bracketed_to_label_offsets(output[0]['text'], entity_type="TEST")


[0, 6, 'TEST']

In [ ]:
from src.codec.translate import process_row

for i in range(22, len(df)):
    ex = df.iloc[i]
    if ex.label:
        print(i)
        target_labels = process_row(
            codec=codec,
            text=ex.text,
            template=ex.source_text,
            labels=ex.label,
            tgt_lang="deu_Latn",
            translate_kwargs=translate_kwargs,
        )

        print(ex.text)
        for s, e, t in ex.label:
            print(f"{t} ({s}-{e})\t{repr(ex.text[s:e])}")

        print(ex.source_text)
        for s, e, t in target_labels:
            print(f"{t} ({s}-{e})\t{repr(ex.source_text[s:e])}")
        break

In [29]:
"Apostar e investir em energias renováveis e sustentáveis, em equilíbrio com o ambiente e articulação com as populações e na gestão pública das empresas do sector energético."[105:118]
"Celle qui aspire â respirer un air encore pur."[0:46]

'Celle qui aspire â respirer un air encore pur.'

## inspect results

In [ ]:
import pandas as pd
from src.codec.translate import labels_to_bracketed
from textwrap import TextWrapper
w = TextWrapper(subsequent_indent="     ", width=70)

out = {}
results = pd.read_json("output2.jsonl", lines=True)
for i, row in results.iterrows():
    if row.label and len(row.label) > 0:
        out[row.id] = {}
        bracketed, _ = labels_to_bracketed(row.text, row.label)
        out[row.id]['english_annotated'] = bracketed
        print(f"\033[92men\033[0m: {w.fill(bracketed)}")
        bracketed, _ = labels_to_bracketed(row.source_text, row.source_label)
        out[row.id]['source_annotated'] = bracketed
        out[row.id]['source_lang'] = row.lang
        print(f"\033[93m{row.lang}\033[0m: {w.fill(bracketed)}")
        print()

: 